# MISTRAL Instrument Walkthrough

This notebook demonstrates how to use the AtLAST Sensitivity Calculator with the MISTRAL (MIllimieter Sardinia radio Telescope Receiver based on Array of Lumped elements KIDs), a KIDs camera observing in the W-band from the Sardinia Radio Telescope

## MISTRAL Specifications

- **Frequency Range**: 77-103 GHz
- **Type**: Lumped elements Kinetic Inductance Detectors (LEKIDs)
- **Purpose**: Wide area mapping
- **Instrument Webpage**: **TBD**
- **Instrument Reference**: [????](URL HERE)

This walkthrough is based on the other AtLAST tutorial notebooks. 

This notebook calculates the sensitivity of MISTRAL based on preliminary data acquired during the commissioning phase at the telescope. We measure the Noise Equivalent Temperature (NET) of our detector array with various atmospheric conditions, building scaling relation between atmospheric opacity $\tau$ and the measured NET. We then convert the NET to NEP using the following formula:

$$ NEP = 2 k_B \Delta\nu NET$$

we then estimate SEFD and system temperature following the same steps as in the TIFUUN walkthrough:

$$ T_{sys} = \frac{NEP}{k_b \eta_{chip}\eta_{co}t\sqrt{2n_{pol}\Delta\nu}}$$

Since the NET is measured using the atmosphere at the SRT, that is an extended source, we are already accounting for the efficiency of detector array and cold optics, so we put these efficiencies to 1. We set $n_{pol}=1$ (in this notation, it means averaging over the two linear polarizations), as MISTRAL is not a polarimeter (yet). 

Note that the NET estimates are **preliminary**. Also, given that this calculator assumes AtLAST's excellent atmosphere, the presented values are a rather extreme extrapolation from SRT's atmospheric conditions to Atacama desert conditions. So, this notebook should not be used to plan actual observations with MISTRAL. 



## Setup and Imports

First, we import the necessary modules and initialise the calculator.

In [1]:
# Import utilities
import numpy as np
import astropy.units as u
from astropy import constants

# Import Sensitivity Calculator
from atlast_sc.calculator import Calculator
from atlast_sc.derived_groups import noise_temperature

# Import Plotting routines
import matplotlib.pyplot as plt
# Set up plotting style
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## Quick Demonstration of the Calculator Usage

Here we show a quick demonstration of how the typical user will interact with the calculator in the MISTRAL parameter space.

MISTRAL observes in a fixed band centered on 90 GHz with a bandwidth of 26 GHz. It is also not sensitive to polarization, i.e. detectors are equally sensitive to both polarizations. 

In [2]:
# Create calculator instance
calculator = Calculator()
print(calculator.chosen_instrument)
# Set observation parameters for MISTRA
calculator.user_input.obs_freq = 90 * u.GHz  # Fixed for MISTRAL
calculator.user_input.bandwidth = 26 * u.GHz  # Fixed for MISTRAL
calculator.user_input.weather = 5.0  # percentile water vapour (see documentation for conversion to mm)
calculator.user_input.t_int = 1.0 * u.s  # on source time only, no calibration time added
calculator.user_input.n_pol = 2 # MISTRAL is dual-pol only

calculator.chosen_instrument = "Mistral"


Default
Instrument has been changed from Default to Mistral.
Tau = 0.029028543561067613
MISTRAL NET= 0.0009827693959919193
MISTRAL NEP= 7.055669835795607e-16
Tsys= 224.10590276990195 K
Tau = 0.028124394404235663
MISTRAL NET= 0.0009575234768453915
MISTRAL NEP= 6.874419920072187e-16
Tsys= 218.34894745091975 K
Tau = 0.028124394404235663
MISTRAL NET= 0.0009575234768453915
MISTRAL NEP= 6.874419920072187e-16
Tsys= 218.34894745091975 K


In [3]:
calculator.user_input.obs_freq

<Quantity 90. GHz>

To show the full list of parameters set for the initial calculations, we can ask the calculator to 'show' the full list of inputs available. Those not explicitly set above are passed their default settings.

In [4]:
print(f"Calculator initialised with the following parameters:")
calculator.user_input.show()

Calculator initialised with the following parameters:
bandwidth: 26.0 GHz
elevation: 45.0 deg
n_pol: 2.0
obs_freq: 90.0 GHz
sensitivity: 3.0 mJy
t_int: 1.0 s
weather: 5.0


### Calculating the Sensitivity or Observing Time for an Observation

Let's try calculating the sensitivity with 1 second of integration time with 25 percentile weather. 

In [5]:
print("T_atm=", calculator.derived_parameters.T_atm)
print("For reference, the T_atm at SRT is typically 280 K.")

T_atm= 236.54843088335838 K
For reference, the T_atm at SRT is typically 280 K.


In [6]:
sens = calculator.calculate_sensitivity()
print(f'RMS sensitivity per independent sky beam: {sens}')

RMS sensitivity per independent sky beam: 1.9306169925113312 mJy


Alternatively we can calculate the on source observing time to reach a given sensitivity.

In [7]:
calculator.user_input.sensitivity = 1*u.mJy
obs_time = calculator.calculate_t_integration()
print(f'On source observing time required: {obs_time}')

On source observing time required: 3.727281971773497 s
